# Orchid ResNet-18 E1
Use a GPU runtime and run all cells. The saved historical baseline reached validation loss 0.51018 at epoch 5 on a different manifest (1,272 validation rows). This sweep uses the pinned branch manifest (1,273 validation rows).

In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys, zipfile
import pandas as pd
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime.'
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive')
dataset_zip = drive_root / 'Colab Notebooks/orchid-project/Orchid Flowers Dataset-v1.1.zip'
if not dataset_zip.is_file():
    raise FileNotFoundError(f'Update dataset_zip to your existing data archive: {dataset_zip}')
repo = Path('/content/orchid-e1')
revision = 'e36dd98'
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'a-data',
                    'https://github.com/satyamchauhan49/orchid-identification.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', revision], check=True)
os.chdir(repo)
manifest = repo / 'outputs/split_manifest_grouped.csv'
frame = pd.read_csv(manifest, dtype={'image_id': str})
assert frame['split'].value_counts().to_dict() == {'train': 4579, 'calib': 509, 'val': 1273, 'test': 795}
assert frame['image_id'].is_unique and frame.groupby('phash_cluster')['split'].nunique().max() == 1
raw = Path('/content/orchid-e1-dataset')
if not list(raw.rglob('Orchid_Images')):
    with zipfile.ZipFile(dataset_zip) as archive:
        archive.extractall(raw)
dataset_root, = [path.parent for path in raw.rglob('Orchid_Images') if path.is_dir()]
assert all((dataset_root / path).is_file() for path in frame['path'])
save_root = drive_root / 'orchid-e1' / revision
save_root.mkdir(parents=True, exist_ok=True)

In [ ]:
rates = {'resnet18_lr_3e-4': '0.0003', 'resnet18_lr_1e-3': '0.001', 'resnet18_lr_1e-4': '0.0001'}
for name, rate in rates.items():
    destination = save_root / name
    if (destination / '.complete').exists():
        print('Reusing', name)
        continue
    subprocess.run([sys.executable, 'models_resnet/train_resnet.py',
                    '--dataset-root', str(dataset_root), '--manifest', str(manifest),
                    '--run-name', name, '--lr', rate, '--epochs', '15', '--batch-size', '32',
                    '--workers', '2', '--seed', '42', '--patience', '4', '--pretrained'], check=True)
    shutil.copytree(repo / 'outputs' / name, destination, dirs_exist_ok=True)
    (destination / '.complete').touch()

In [ ]:
import json
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score

val = frame.loc[frame['split'] == 'val'].set_index('image_id')
shared_maps = json.loads((repo / 'outputs/label_maps.json').read_text())
heads = ('species', 'T', 'NF', 'In', 'LC', 'CF', 'CL')
summary, details = [], []
for name, rate in rates.items():
    run = save_root / name
    history = pd.read_csv(run / 'training_history.csv')
    best = history.loc[history['val_loss'].idxmin()]
    maps = json.loads((run / 'class_maps.json').read_text())
    with np.load(run / 'val.npz', allow_pickle=False) as data:
        ids = pd.Index(data['ids'].astype(str))
        assert ids.is_unique and set(ids) == set(val.index)
        for head in heads:
            labels = sorted(maps[head], key=maps[head].get)
            assert labels == [str(label) for label in shared_maps[head]], head
            y = data[f'targets_{head}']
            assert np.array_equal(y, val.loc[ids, head].astype(str).map(maps[head]).to_numpy())
            key = 'species_logits' if head == 'species' else f'logits_{head}'
            logits = data[key]
            assert logits.shape == (len(y), len(labels)) and np.isfinite(logits).all()
            ranked = np.argsort(-logits, axis=1, kind='stable')
            predicted = ranked[:, 0]
            metrics = dict(run=name, learning_rate=float(rate), head=head, rows=len(y),
                           best_epoch=int(best['epoch']), val_loss=float(best['val_loss']),
                           top1=(predicted == y).mean(), macro_f1=f1_score(y, predicted, average='macro'))
            if head == 'species':
                for k in (3, 5):
                    metrics[f'top{k}'] = (ranked[:, :k] == y[:, None]).any(axis=1).mean()
            else:
                cm = confusion_matrix(y, predicted, labels=range(len(labels)))
                pd.DataFrame(cm, index=labels, columns=labels).rename_axis('actual').to_csv(
                    run / f'val_confusion_{head}.csv')
                for i, label in enumerate(labels):
                    tp, positives = int(cm[i, i]), int(cm[i, :].sum())
                    fp, negatives = int(cm[:, i].sum()) - tp, len(y) - positives
                    details.append(dict(run=name, head=head, label=label, support=positives,
                                        tpr=tp / positives if positives else np.nan,
                                        tnr=(negatives - fp) / negatives if negatives else np.nan))
            summary.append(metrics)
summary, details = pd.DataFrame(summary), pd.DataFrame(details)
summary.to_csv(save_root / 'R1_validation_summary.csv', index=False)
details.to_csv(save_root / 'R1_validation_by_class.csv', index=False)
winner = summary.query("head == 'species'").sort_values(['val_loss', 'learning_rate']).iloc[0]
selected = save_root / 'resnet18_base'
shutil.copytree(save_root / winner['run'], selected, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('.complete'))
(selected / 'selection.json').write_text(json.dumps(winner.drop('head').to_dict(), indent=2))
display(summary)
display(details.loc[details['label'].isin(['GreenYellow', 'Panicle', 'WhiteYellow'])])
print('Selected baseline for team handoff:', selected)

R1 metrics use validation only. Blank rates indicate a zero denominator. The lowest validation loss selects the baseline. Share the selected baseline folder through Drive and leave test logits unopened until locked evaluation.